# Direct Preference Optimization (DPO)

## Overview

Direct Preference Optimization (DPO) is a simpler alternative to RLHF that directly optimizes language models from preference data without reinforcement learning.

### Historical Context
- **2022**: RLHF becomes standard (InstructGPT, ChatGPT)
- **May 2023**: DPO paper released (Rafailov et al., Stanford)
- **2023-2024**: Rapid adoption due to simplicity
- **Today**: Used by Zephyr, Starling, and many open models

### Key Insight

RLHF has a complex pipeline:
```
SFT → Reward Model → PPO (with 4 models)
```

DPO simplifies to:
```
SFT → DPO (with 2 models)
```

### Why DPO is Simpler

1. **No reward model**: Optimizes preferences directly
2. **No RL**: Simple classification loss
3. **Fewer models**: Just policy + reference
4. **More stable**: No PPO hyperparameters
5. **Efficient**: Faster training, less memory

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
from tqdm import tqdm
import math

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. The DPO Paper (2023)

### Motivation

**RLHF Challenges**:
1. Reward model can be inaccurate
2. PPO is complex and unstable
3. Four models needed simultaneously
4. Hyperparameter sensitive

**DPO Question**: Can we skip the reward model and RL, optimizing preferences directly?

### Key Contributions

1. **Theoretical**: Derived closed-form mapping from reward to optimal policy
2. **Practical**: Simple loss function for direct optimization
3. **Empirical**: Matches or beats RLHF on multiple tasks
4. **Efficient**: 2-3x faster, lower memory

### Paper Results

Tested on:
- **Sentiment control**: 58% → 72% positive rate
- **Summarization**: Better ROUGE scores
- **Dialogue**: Higher human preference win rate
- **Anthropic Helpfulness**: Comparable to RLHF

## 2. Mathematical Derivation

### RLHF Objective

RLHF maximizes reward with KL constraint:

$$\max_{\pi_\theta} \mathbb{E}_{x \sim D, y \sim \pi_\theta(y|x)} [r(x, y)] - \beta \cdot D_{KL}(\pi_\theta(y|x) || \pi_{ref}(y|x))$$

This has a closed-form optimal solution:

$$\pi^*(y|x) = \frac{1}{Z(x)} \pi_{ref}(y|x) \exp\left(\frac{1}{\beta} r(x, y)\right)$$

where $Z(x)$ is the partition function.

### Reparameterization

We can solve for the reward:

$$r(x, y) = \beta \log \frac{\pi^*(y|x)}{\pi_{ref}(y|x)} + \beta \log Z(x)$$

### Bradley-Terry Model

Human preferences follow:

$$P(y_w \succ y_l | x) = \frac{\exp(r(x, y_w))}{\exp(r(x, y_w)) + \exp(r(x, y_l))}$$

### DPO Derivation

Substituting the reward reparameterization:

$$P(y_w \succ y_l | x) = \frac{1}{1 + \exp\left(\beta \log \frac{\pi^*(y_l|x)}{\pi_{ref}(y_l|x)} - \beta \log \frac{\pi^*(y_w|x)}{\pi_{ref}(y_w|x)}\right)}$$

Simplifying:

$$P(y_w \succ y_l | x) = \sigma\left(\beta \log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\right)$$

### DPO Loss

Maximize log-likelihood of preferences:

$$\mathcal{L}_{DPO}(\theta) = -\mathbb{E}_{(x,y_w,y_l) \sim D} \left[ \log \sigma\left(\beta \log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\right) \right]$$

### Key Insight

The loss directly compares policy log-probs, no reward model needed!

The partition function $Z(x)$ cancels out in the ratio.

In [ ]:
def visualize_dpo_derivation():
    """Visualize the key mathematical relationships in DPO."""
    
    # Log probability ratios
    log_ratios = np.linspace(-3, 3, 100)
    beta = 0.1
    
    # Implicit reward (from reparameterization)
    implicit_rewards = beta * log_ratios
    
    # Preference probability
    probs = 1 / (1 + np.exp(-implicit_rewards))
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Implicit reward vs log-ratio
    ax1.plot(log_ratios, implicit_rewards, linewidth=2)
    ax1.axhline(y=0, color='r', linestyle='--', alpha=0.5)
    ax1.axvline(x=0, color='r', linestyle='--', alpha=0.5)
    ax1.set_xlabel(r'$\log \frac{\pi_\theta(y|x)}{\pi_{ref}(y|x)}$', fontsize=14)
    ax1.set_ylabel(r'Implicit Reward $r(x,y)$', fontsize=14)
    ax1.set_title('DPO Reparameterization', fontsize=14)
    ax1.grid(True, alpha=0.3)
    ax1.text(1.5, 0.15, r'$r(x,y) = \beta \log \frac{\pi}{\pi_{ref}}$', 
             fontsize=12, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # Preference probability
    ax2.plot(log_ratios, probs, linewidth=2, color='green')
    ax2.axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Neutral')
    ax2.axvline(x=0, color='r', linestyle='--', alpha=0.5)
    ax2.set_xlabel(r'Log-ratio Difference $\Delta$', fontsize=14)
    ax2.set_ylabel(r'$P(y_w \succ y_l)$', fontsize=14)
    ax2.set_title('Preference Probability', fontsize=14)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n=== DPO Key Insight ===")
    print("\n1. Reward is reparameterized as log-ratio of policy/reference")
    print("2. Partition function Z(x) cancels in preference probability")
    print("3. Can optimize preferences without explicit reward model")
    print("4. Loss is simple: maximize likelihood of preferences")

visualize_dpo_derivation()

## 3. DPO Implementation from Scratch

### Components

1. **Policy model** $\pi_\theta$: The model we're training
2. **Reference model** $\pi_{ref}$: Frozen copy (typically SFT model)
3. **Preference dataset**: $(x, y_{chosen}, y_{rejected})$ triples

### Algorithm

```python
# Initialize
policy_model = load_sft_model()
ref_model = copy(policy_model).freeze()

# Training loop
for batch in dataloader:
    x, y_w, y_l = batch
    
    # Compute log probabilities
    logp_w_policy = policy_model.log_prob(y_w | x)
    logp_l_policy = policy_model.log_prob(y_l | x)
    logp_w_ref = ref_model.log_prob(y_w | x)
    logp_l_ref = ref_model.log_prob(y_l | x)
    
    # Compute DPO loss
    logits = beta * (logp_w_policy - logp_w_ref) - beta * (logp_l_policy - logp_l_ref)
    loss = -F.logsigmoid(logits).mean()
    
    # Optimize
    loss.backward()
    optimizer.step()
```

In [ ]:
class SimpleLanguageModel(nn.Module):
    """Simple transformer for DPO demonstration."""
    
    def __init__(
        self,
        vocab_size: int,
        hidden_dim: int = 512,
        num_layers: int = 6,
        num_heads: int = 8,
        max_seq_len: int = 512,
        dropout: float = 0.1
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.pos_embedding = nn.Embedding(max_seq_len, hidden_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=4 * hidden_dim,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        self.lm_head = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)
        
    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """
        Args:
            input_ids: (batch_size, seq_len)
            attention_mask: (batch_size, seq_len)
        
        Returns:
            logits: (batch_size, seq_len, vocab_size)
        """
        batch_size, seq_len = input_ids.shape
        
        # Embeddings
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        x = self.embedding(input_ids) + self.pos_embedding(positions)
        x = self.dropout(x)
        
        # Transformer
        if attention_mask is not None:
            attention_mask = attention_mask.bool()
        
        x = self.transformer(x, src_key_padding_mask=~attention_mask if attention_mask is not None else None)
        
        # Language modeling head
        logits = self.lm_head(x)
        
        return logits
    
    def get_log_probs(
        self,
        input_ids: torch.Tensor,
        labels: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """
        Get log probabilities for specific tokens.
        
        Args:
            input_ids: (batch_size, seq_len) input tokens
            labels: (batch_size, seq_len) target tokens
            attention_mask: (batch_size, seq_len)
        
        Returns:
            log_probs: (batch_size,) average log prob per sequence
        """
        logits = self.forward(input_ids, attention_mask)
        log_probs = F.log_softmax(logits, dim=-1)
        
        # Gather log probs for labels
        gathered_log_probs = log_probs.gather(-1, labels.unsqueeze(-1)).squeeze(-1)
        
        # Average over sequence (with masking)
        if attention_mask is not None:
            gathered_log_probs = gathered_log_probs * attention_mask
            avg_log_probs = gathered_log_probs.sum(dim=1) / attention_mask.sum(dim=1)
        else:
            avg_log_probs = gathered_log_probs.mean(dim=1)
        
        return avg_log_probs


# Test model
model = SimpleLanguageModel(vocab_size=1000, hidden_dim=256, num_layers=4)
print(f"Model Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Test forward pass
batch_size, seq_len = 4, 32
input_ids = torch.randint(0, 1000, (batch_size, seq_len))
labels = torch.randint(0, 1000, (batch_size, seq_len))
attention_mask = torch.ones(batch_size, seq_len)

logits = model(input_ids, attention_mask)
log_probs = model.get_log_probs(input_ids, labels, attention_mask)

print(f"\nLogits shape: {logits.shape}")
print(f"Log probs shape: {log_probs.shape}")
print(f"Sample log probs: {log_probs}")

In [ ]:
class DPOLoss(nn.Module):
    """Direct Preference Optimization loss."""
    
    def __init__(self, beta: float = 0.1, label_smoothing: float = 0.0):
        super().__init__()
        self.beta = beta
        self.label_smoothing = label_smoothing
    
    def forward(
        self,
        policy_chosen_logps: torch.Tensor,
        policy_rejected_logps: torch.Tensor,
        reference_chosen_logps: torch.Tensor,
        reference_rejected_logps: torch.Tensor,
    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        """
        Compute DPO loss.
        
        Args:
            policy_chosen_logps: (batch_size,) log P(y_chosen | x) under policy
            policy_rejected_logps: (batch_size,) log P(y_rejected | x) under policy
            reference_chosen_logps: (batch_size,) log P(y_chosen | x) under reference
            reference_rejected_logps: (batch_size,) log P(y_rejected | x) under reference
        
        Returns:
            loss: scalar loss
            metrics: dictionary of metrics
        """
        # Compute log-ratios
        pi_logratios = policy_chosen_logps - policy_rejected_logps
        ref_logratios = reference_chosen_logps - reference_rejected_logps
        
        # DPO logits (implicit rewards)
        logits = self.beta * (pi_logratios - ref_logratios)
        
        # Loss: -log sigmoid(logits)
        if self.label_smoothing > 0:
            # Label smoothing: interpolate with uniform
            loss = -F.logsigmoid(logits) * (1 - self.label_smoothing) - F.logsigmoid(-logits) * self.label_smoothing
        else:
            loss = -F.logsigmoid(logits)
        
        loss = loss.mean()
        
        # Compute metrics
        with torch.no_grad():
            # Implicit rewards
            chosen_rewards = self.beta * (policy_chosen_logps - reference_chosen_logps)
            rejected_rewards = self.beta * (policy_rejected_logps - reference_rejected_logps)
            
            # Accuracy: % of times chosen has higher reward
            accuracy = (chosen_rewards > rejected_rewards).float().mean()
            
            metrics = {
                'loss': loss.item(),
                'accuracy': accuracy.item(),
                'chosen_reward_mean': chosen_rewards.mean().item(),
                'rejected_reward_mean': rejected_rewards.mean().item(),
                'reward_margin': (chosen_rewards - rejected_rewards).mean().item(),
                'logits_mean': logits.mean().item(),
            }
        
        return loss, metrics


# Test DPO loss
criterion = DPOLoss(beta=0.1)

# Simulate log probabilities
policy_chosen = torch.tensor([-2.0, -1.5, -1.8])
policy_rejected = torch.tensor([-3.0, -2.5, -2.8])
ref_chosen = torch.tensor([-2.2, -1.6, -1.9])
ref_rejected = torch.tensor([-2.8, -2.4, -2.7])

loss, metrics = criterion(policy_chosen, policy_rejected, ref_chosen, ref_rejected)

print("\n=== DPO Loss Test ===")
print(f"Loss: {loss.item():.4f}")
print(f"\nMetrics:")
for key, value in metrics.items():
    print(f"  {key}: {value:.4f}")

## 4. Training with DPO

### Training Procedure

1. **Initialize**: Load SFT model as both policy and reference
2. **Freeze reference**: Reference model stays fixed
3. **Train policy**: Update only policy model with DPO loss
4. **Monitor**: Track rewards and accuracy

### Hyperparameters

- **Beta ($\beta$)**: 0.1 - 0.5
  - Higher β: stronger KL constraint
  - Lower β: more aggressive optimization
- **Learning rate**: 1e-6 to 5e-6
- **Batch size**: 64-256
- **Epochs**: 1-3
- **Label smoothing**: 0-0.1 (optional)

In [ ]:
@dataclass
class DPOExample:
    """Single DPO training example."""
    prompt: str
    chosen: str
    rejected: str


class DPODataset(Dataset):
    """Dataset for DPO training."""
    
    def __init__(
        self,
        examples: List[DPOExample],
        tokenizer: Optional[callable] = None,
        max_length: int = 512
    ):
        self.examples = examples
        self.tokenizer = tokenizer or self._simple_tokenizer
        self.max_length = max_length
    
    def _simple_tokenizer(self, text: str) -> List[int]:
        """Simple character-level tokenizer."""
        return [ord(c) % 1000 for c in text[:self.max_length]]
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        example = self.examples[idx]
        
        # Tokenize
        chosen_text = example.prompt + " " + example.chosen
        rejected_text = example.prompt + " " + example.rejected
        
        chosen_ids = self.tokenizer(chosen_text)
        rejected_ids = self.tokenizer(rejected_text)
        
        # Pad to same length
        max_len = max(len(chosen_ids), len(rejected_ids))
        chosen_ids = chosen_ids + [0] * (max_len - len(chosen_ids))
        rejected_ids = rejected_ids + [0] * (max_len - len(rejected_ids))
        
        chosen_mask = [1] * len(self.tokenizer(chosen_text)) + [0] * (max_len - len(self.tokenizer(chosen_text)))
        rejected_mask = [1] * len(self.tokenizer(rejected_text)) + [0] * (max_len - len(self.tokenizer(rejected_text)))
        
        return {
            'chosen_input_ids': torch.tensor(chosen_ids),
            'chosen_labels': torch.tensor(chosen_ids),
            'chosen_attention_mask': torch.tensor(chosen_mask),
            'rejected_input_ids': torch.tensor(rejected_ids),
            'rejected_labels': torch.tensor(rejected_ids),
            'rejected_attention_mask': torch.tensor(rejected_mask),
        }


def train_dpo(
    policy_model: nn.Module,
    ref_model: nn.Module,
    train_loader: DataLoader,
    val_loader: Optional[DataLoader] = None,
    num_epochs: int = 3,
    learning_rate: float = 1e-5,
    beta: float = 0.1,
    device: torch.device = torch.device('cpu')
) -> Dict[str, List[float]]:
    """Train model with DPO."""
    
    policy_model = policy_model.to(device)
    ref_model = ref_model.to(device)
    
    # Freeze reference model
    for param in ref_model.parameters():
        param.requires_grad = False
    ref_model.eval()
    
    criterion = DPOLoss(beta=beta)
    optimizer = torch.optim.AdamW(policy_model.parameters(), lr=learning_rate)
    
    history = {'train_loss': [], 'train_accuracy': [], 'reward_margin': []}
    
    for epoch in range(num_epochs):
        policy_model.train()
        epoch_metrics = {'loss': [], 'accuracy': [], 'reward_margin': []}
        
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')
        for batch in pbar:
            # Move to device
            chosen_ids = batch['chosen_input_ids'].to(device)
            chosen_labels = batch['chosen_labels'].to(device)
            chosen_mask = batch['chosen_attention_mask'].to(device)
            rejected_ids = batch['rejected_input_ids'].to(device)
            rejected_labels = batch['rejected_labels'].to(device)
            rejected_mask = batch['rejected_attention_mask'].to(device)
            
            # Get log probabilities from policy
            policy_chosen_logps = policy_model.get_log_probs(chosen_ids, chosen_labels, chosen_mask)
            policy_rejected_logps = policy_model.get_log_probs(rejected_ids, rejected_labels, rejected_mask)
            
            # Get log probabilities from reference (frozen)
            with torch.no_grad():
                ref_chosen_logps = ref_model.get_log_probs(chosen_ids, chosen_labels, chosen_mask)
                ref_rejected_logps = ref_model.get_log_probs(rejected_ids, rejected_labels, rejected_mask)
            
            # Compute DPO loss
            loss, metrics = criterion(
                policy_chosen_logps,
                policy_rejected_logps,
                ref_chosen_logps,
                ref_rejected_logps
            )
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy_model.parameters(), 1.0)
            optimizer.step()
            
            # Track metrics
            epoch_metrics['loss'].append(metrics['loss'])
            epoch_metrics['accuracy'].append(metrics['accuracy'])
            epoch_metrics['reward_margin'].append(metrics['reward_margin'])
            
            pbar.set_postfix({
                'loss': f"{metrics['loss']:.4f}",
                'acc': f"{metrics['accuracy']:.3f}",
                'margin': f"{metrics['reward_margin']:.3f}"
            })
        
        # Epoch summary
        history['train_loss'].append(np.mean(epoch_metrics['loss']))
        history['train_accuracy'].append(np.mean(epoch_metrics['accuracy']))
        history['reward_margin'].append(np.mean(epoch_metrics['reward_margin']))
        
        print(f"\nEpoch {epoch+1}: Loss={history['train_loss'][-1]:.4f}, "
              f"Accuracy={history['train_accuracy'][-1]:.3f}, "
              f"Margin={history['reward_margin'][-1]:.3f}")
    
    return history


# Create synthetic dataset
def create_dpo_dataset(num_examples: int = 500) -> List[DPOExample]:
    """Create synthetic DPO examples."""
    examples = []
    
    prompts = [
        "Explain quantum mechanics:",
        "Write a haiku:",
        "What is democracy?",
        "Describe photosynthesis:",
        "Tell me about AI:"
    ]
    
    for i in range(num_examples):
        prompt = np.random.choice(prompts)
        chosen = f"High quality detailed response {i} with good explanation."
        rejected = f"Low quality brief response {i}."
        
        examples.append(DPOExample(prompt, chosen, rejected))
    
    return examples


print("DPO training setup ready!")
print("\nKey differences from RLHF:")
print("- No reward model needed")
print("- No PPO complexity")
print("- Just 2 models (policy + reference)")
print("- Simple classification loss")

In [ ]:
# Train DPO model
print("Training DPO model on synthetic data...\n")

# Create dataset
dpo_examples = create_dpo_dataset(500)
dataset = DPODataset(dpo_examples)

# Split data
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

# Initialize models
policy_model = SimpleLanguageModel(vocab_size=1000, hidden_dim=128, num_layers=3)
ref_model = SimpleLanguageModel(vocab_size=1000, hidden_dim=128, num_layers=3)

# Copy weights from policy to reference
ref_model.load_state_dict(policy_model.state_dict())

# Train
history = train_dpo(
    policy_model,
    ref_model,
    train_loader,
    val_loader,
    num_epochs=3,
    learning_rate=1e-5,
    beta=0.1,
    device=device
)

In [ ]:
def plot_dpo_training(history: Dict[str, List[float]]):
    """Plot DPO training curves."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Loss
    axes[0].plot(epochs, history['train_loss'], 'o-', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title('DPO Loss', fontsize=14)
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1].plot(epochs, history['train_accuracy'], 'o-', color='green', linewidth=2)
    axes[1].axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Random')
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Accuracy', fontsize=12)
    axes[1].set_title('Preference Accuracy', fontsize=14)
    axes[1].set_ylim(0, 1)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Reward margin
    axes[2].plot(epochs, history['reward_margin'], 'o-', color='orange', linewidth=2)
    axes[2].axhline(y=0, color='r', linestyle='--', alpha=0.5)
    axes[2].set_xlabel('Epoch', fontsize=12)
    axes[2].set_ylabel('Reward Margin', fontsize=12)
    axes[2].set_title('Chosen - Rejected Reward', fontsize=14)
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_dpo_training(history)

## 5. DPO vs RLHF Comparison

### Complexity

| Aspect | RLHF | DPO |
|--------|------|-----|
| Models | 4 (policy, value, reward, reference) | 2 (policy, reference) |
| Training | PPO (complex) | Supervised (simple) |
| Hyperparameters | Many (clip, KL, GAE, etc.) | Few (beta, LR) |
| Stability | Can be unstable | More stable |
| Memory | High (4 models) | Lower (2 models) |
| Speed | Slower (rollouts) | Faster (direct) |

### Performance

**RLHF Advantages**:
- More expressive (separate reward model)
- Can handle sparse rewards
- Better for complex tasks
- Industry-proven at scale

**DPO Advantages**:
- Simpler to implement
- More stable training
- Faster convergence
- Lower memory footprint
- Easier hyperparameter tuning

### Empirical Results

From the DPO paper:
- **Sentiment**: DPO matches RLHF performance
- **Summarization**: DPO slightly better on ROUGE
- **Dialogue**: Comparable human preference win rates
- **Efficiency**: 2-3x faster, 40% less memory

In [ ]:
def compare_rlhf_dpo():
    """Visualize RLHF vs DPO comparison."""
    
    categories = ['Simplicity', 'Stability', 'Speed', 'Memory\nEfficiency', 'Implementation\nTime']
    rlhf_scores = [3, 5, 4, 4, 3]
    dpo_scores = [9, 8, 8, 8, 9]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Comparison radar
    x = np.arange(len(categories))
    width = 0.35
    
    bars1 = ax1.bar(x - width/2, rlhf_scores, width, label='RLHF', alpha=0.8)
    bars2 = ax1.bar(x + width/2, dpo_scores, width, label='DPO', alpha=0.8)
    
    ax1.set_ylabel('Score (0-10)', fontsize=12)
    ax1.set_title('Practical Comparison: RLHF vs DPO', fontsize=14)
    ax1.set_xticks(x)
    ax1.set_xticklabels(categories, rotation=45, ha='right')
    ax1.legend()
    ax1.set_ylim(0, 10)
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Complexity flow diagram
    ax2.text(0.5, 0.9, 'RLHF Pipeline', ha='center', fontsize=14, 
             weight='bold', transform=ax2.transAxes)
    ax2.text(0.5, 0.75, 'SFT → Reward Model → PPO', ha='center', fontsize=11,
             family='monospace', transform=ax2.transAxes,
             bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.5))
    ax2.text(0.5, 0.60, '(4 models, complex)', ha='center', fontsize=10,
             style='italic', transform=ax2.transAxes)
    
    ax2.text(0.5, 0.4, 'DPO Pipeline', ha='center', fontsize=14,
             weight='bold', transform=ax2.transAxes)
    ax2.text(0.5, 0.25, 'SFT → DPO', ha='center', fontsize=11,
             family='monospace', transform=ax2.transAxes,
             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))
    ax2.text(0.5, 0.10, '(2 models, simple)', ha='center', fontsize=10,
             style='italic', transform=ax2.transAxes)
    
    ax2.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print("\n=== RLHF vs DPO ===")
    print("\nRLHF: More complex but flexible")
    print("  + Separate reward model")
    print("  + Industry proven")
    print("  - Complex PPO algorithm")
    print("  - Requires 4 models")
    print("\nDPO: Simpler and faster")
    print("  + Direct optimization")
    print("  + Only 2 models needed")
    print("  + More stable training")
    print("  - Implicit reward function")

compare_rlhf_dpo()

## 6. When to Use DPO vs PPO

### Choose DPO when:

1. **Limited computational resources**
   - Smaller models (< 13B parameters)
   - Single GPU training
   - Memory constraints

2. **Quick iteration**
   - Research prototyping
   - Fast experiments
   - Simple alignment tasks

3. **Stable training preferred**
   - Less hyperparameter tuning needed
   - More predictable behavior
   - Fewer failure modes

4. **Direct preferences available**
   - Clean pairwise comparisons
   - Good quality preference data

### Choose RLHF/PPO when:

1. **Large-scale deployment**
   - Very large models (> 70B)
   - Production systems
   - Industry-proven pipeline

2. **Complex reward functions**
   - Multi-objective optimization
   - Compositional rewards
   - Separate reward modeling needed

3. **Sparse or dense rewards**
   - Non-comparison feedback
   - Intermediate rewards
   - Complex task decomposition

4. **Maximum performance**
   - Willing to invest in tuning
   - Have computational resources
   - Need slight edge in quality

### Hybrid Approaches

Some recent work combines both:
- **Start with DPO**: Quick initial alignment
- **Refine with PPO**: Polish with complex rewards
- **Iterative**: Alternate between DPO and PPO

In [ ]:
def visualize_method_selection():
    """Decision tree for choosing DPO vs RLHF."""
    
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Decision tree
    decisions = [
        (0.5, 0.95, 'Start: Need to align LLM', 'black', 'bold'),
        (0.3, 0.75, 'Limited\nResources?', 'blue', 'normal'),
        (0.7, 0.75, 'Complex\nRewards?', 'blue', 'normal'),
        (0.15, 0.55, 'Use DPO', 'green', 'bold'),
        (0.45, 0.55, 'Try DPO first', 'green', 'normal'),
        (0.7, 0.55, 'Need max\nperformance?', 'blue', 'normal'),
        (0.6, 0.35, 'Use DPO', 'green', 'bold'),
        (0.85, 0.35, 'Use RLHF/PPO', 'orange', 'bold'),
    ]
    
    for x, y, text, color, weight in decisions:
        bbox_props = dict(boxstyle='round,pad=0.5', 
                          facecolor='lightgreen' if color == 'green' else 'lightcoral' if color == 'orange' else 'lightblue',
                          alpha=0.7)
        ax.text(x, y, text, ha='center', va='center', fontsize=11,
                weight=weight, bbox=bbox_props)
    
    # Arrows
    arrows = [
        ((0.5, 0.90), (0.3, 0.80)),
        ((0.5, 0.90), (0.7, 0.80)),
        ((0.3, 0.70), (0.15, 0.60)),
        ((0.3, 0.70), (0.45, 0.60)),
        ((0.7, 0.70), (0.7, 0.60)),
        ((0.7, 0.50), (0.6, 0.40)),
        ((0.7, 0.50), (0.85, 0.40)),
    ]
    
    for start, end in arrows:
        ax.annotate('', xy=end, xytext=start,
                    arrowprops=dict(arrowstyle='->', lw=2, color='gray'))
    
    # Labels on arrows
    labels = [
        (0.4, 0.85, 'Yes'),
        (0.6, 0.85, 'No'),
        (0.22, 0.73, 'Yes'),
        (0.37, 0.73, 'No'),
        (0.65, 0.62, 'Yes'),
        (0.75, 0.62, 'No'),
    ]
    
    for x, y, label in labels:
        ax.text(x, y, label, ha='center', fontsize=9, style='italic', color='red')
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0.2, 1)
    ax.axis('off')
    ax.set_title('Choosing Between DPO and RLHF', fontsize=16, weight='bold', pad=20)
    
    plt.tight_layout()
    plt.show()

visualize_method_selection()

## 7. Practical Examples

### Example 1: Using DPO with Transformers

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOTrainer, DPOConfig

# Load model
model = AutoModelForCausalLM.from_pretrained('gpt2')
ref_model = AutoModelForCausalLM.from_pretrained('gpt2')
tokenizer = AutoTokenizer.from_pretrained('gpt2')

# Configure DPO
config = DPOConfig(
    beta=0.1,
    learning_rate=5e-7,
    per_device_train_batch_size=4,
    num_train_epochs=3,
    max_length=512,
)

# Initialize trainer
trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,
    config=config,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
)

# Train
trainer.train()
```

### Example 2: Custom DPO Loss

```python
# Add length penalty to DPO
class DPOWithLengthPenalty(DPOLoss):
    def __init__(self, beta=0.1, length_penalty=0.01):
        super().__init__(beta)
        self.length_penalty = length_penalty
    
    def forward(self, policy_chosen_logps, policy_rejected_logps,
                ref_chosen_logps, ref_rejected_logps,
                chosen_lengths, rejected_lengths):
        # Standard DPO loss
        loss, metrics = super().forward(
            policy_chosen_logps, policy_rejected_logps,
            ref_chosen_logps, ref_rejected_logps
        )
        
        # Add length penalty
        length_diff = (chosen_lengths - rejected_lengths).abs().float()
        length_loss = self.length_penalty * length_diff.mean()
        
        total_loss = loss + length_loss
        metrics['length_loss'] = length_loss.item()
        
        return total_loss, metrics
```

### Example 3: Zephyr 7B (DPO Success Story)

Zephyr-7B-beta achieved strong performance using:
1. **Base**: Mistral-7B
2. **SFT**: UltraChat (200k examples)
3. **DPO**: UltraFeedback (62k preferences)
4. **Result**: Outperforms Llama-2-70B-chat on MT-Bench

Training time: ~10 hours on 8xA100 GPUs

## Summary

### Key Takeaways

1. **DPO** is a simpler alternative to RLHF that optimizes preferences directly

2. **Mathematical insight**: Reparameterize rewards as log-ratios, partition function cancels

3. **Advantages**: Simpler, more stable, faster, lower memory than RLHF

4. **Loss function**: Compare policy log-probs vs reference, weighted by beta

5. **Performance**: Matches RLHF on many tasks, especially with good preference data

6. **When to use**: Limited resources, quick iteration, stable training preferred

7. **Real-world success**: Zephyr, Starling, and many open models use DPO

### Limitations

1. **Implicit rewards**: Can't inspect or debug reward function
2. **Less flexible**: Harder to add complex objectives
3. **Preference data**: Requires clean pairwise comparisons
4. **Beta sensitivity**: Still needs some tuning

### Future Directions

- **IPO** (Identity Preference Optimization): Removes sigmoid
- **KTO** (Kahneman-Tversky Optimization): Uses single examples
- **ORPO** (Odds Ratio Preference Optimization): Combines SFT and DPO
- **Hybrid methods**: DPO + PPO combinations

### Next Steps

- **Notebook 43**: Constitutional AI for self-improvement
- **Notebook 44**: Domain adaptation techniques

### References

1. Rafailov et al. (2023): "Direct Preference Optimization: Your Language Model is Secretly a Reward Model"
2. Tunstall et al. (2023): "Zephyr: Direct Distillation of LM Alignment"
3. Ouyang et al. (2022): "Training language models to follow instructions with human feedback"
4. Ethayarajh et al. (2024): "KTO: Model Alignment as Prospect Theoretic Optimization"
5. Hong et al. (2024): "ORPO: Monolithic Preference Optimization without Reference Model"